# 04 -- Luck Score Demo (time-ordered, real-data)

**Contact Luck Prototype v0.1**

Walks through the full scoring pipeline for a single 2024 play, using the **unweighted** probability baseline selected in `03_calibration.ipynb` (see README.md "Model comparison and probability calibration" for why -- `class_weight="balanced"` was confirmed to produce severely miscalibrated probabilities and must never be used here).

This notebook is strictly time-ordered:
- The model trains ONLY on `TRAIN_SEASONS` (2021-2023).
- The example play is selected ONLY from `VALIDATION_SEASONS` (2024).
- The example play (and every other 2024 row) is never part of training.
- 2025 is never loaded or used.

If the full development dataset is missing, this notebook falls back to a small synthetic example (never to the one-week sample -- see the data-loading cell for why) so the pipeline is always runnable.

> **This repository is a research prototype, not a validated public baseball statistic.** See `README.md` and `CLAUDE.md` for full scope and limitations.

In [1]:
import pandas as pd

from mlb_luck_score.config import (
    CLASS_ORDER,
    DEFAULT_VALUE_MAP,
    PROCESSED_DATA_DIR,
    TRAIN_SEASONS,
    VALIDATION_SEASONS,
    assert_seasons_allowed,
)
from mlb_luck_score.models.predict_outcomes import _build_synthetic_demo_data
from mlb_luck_score.models.train_contact_model import predict_proba_ordered, train_model
from mlb_luck_score.scoring.confidence import compute_confidence
from mlb_luck_score.scoring.public_score import raw_luck_to_public_score
from mlb_luck_score.scoring.raw_luck import compute_expected_value, compute_raw_luck

DEVELOPMENT_PATH = PROCESSED_DATA_DIR / "cleaned_development_data.parquet"
SAMPLE_PATH = PROCESSED_DATA_DIR / "cleaned_batted_balls.parquet"
pd.set_option("display.width", 120)

## Load data

Prefers the full 2021-2024 development dataset. Falls back to a small synthetic example ONLY when that dataset is missing -- **never** to the one-week sample, because the sample covers just 7 days of 2024 and cannot satisfy the required 2021-2023 train / 2024 validation time-ordering (training would have 0 rows). If the sample happens to exist, this cell says so explicitly rather than silently using it.

In [2]:
using_synthetic = False
if DEVELOPMENT_PATH.exists():
    df = pd.read_parquet(DEVELOPMENT_PATH)
    print(f"Loaded the full development dataset: {len(df)} rows from {DEVELOPMENT_PATH}")
else:
    df = None
    using_synthetic = True
    if SAMPLE_PATH.exists():
        print(
            "WARNING: full development dataset not found, but the one-week sample "
            f"exists at {SAMPLE_PATH}. This notebook does NOT fall back to it -- the "
            "sample only covers 7 days of 2024 and cannot provide any 2021-2023 "
            "training rows, which would break the time-ordered design this notebook "
            "demonstrates. Falling back to synthetic demo data instead.\n"
            "Run `make download-development-data` then `make clean-development-data` "
            "for a real 2021-2024 walkthrough."
        )
    else:
        print(
            f"No full development dataset found at {DEVELOPMENT_PATH}.\n"
            "Falling back to synthetic demo data. Run `make download-development-data` "
            "then `make clean-development-data` for a real 2021-2024 walkthrough."
        )

Loaded the full development dataset: 494173 rows from /Users/arihantaneja/Downloads/TrueLuckMLBStat/data/processed/cleaned_development_data.parquet


## Filter to training-eligible rows, guard 2025, and split by season

`eligible_for_training` excludes ambiguous field-error/fielder's-choice rows and rows with missing required contact data (see `mlb_luck_score.eligibility`). The explicit `assert_seasons_allowed` call below is a defense-in-depth check: this notebook only ever requests `TRAIN_SEASONS`/`VALIDATION_SEASONS` (2021-2024), so 2025 should never appear, but this makes that guarantee loud and explicit rather than assumed.

In [3]:
if using_synthetic:
    # Fabricated data has no real seasons -- assign a synthetic season split so the
    # SAME time-ordered code path below runs for both real and synthetic data. This
    # does not exercise real time-ordering (there's nothing to order), it only keeps
    # the notebook runnable without a second code path.
    training_eligible = _build_synthetic_demo_data()
    training_eligible["eligible_for_training"] = True
    training_eligible["season"] = TRAIN_SEASONS[0]
    training_eligible.loc[training_eligible.index[-1:], "season"] = VALIDATION_SEASONS[0]
    for col in ("game_date", "batter", "pitcher", "description", "player_name"):
        if col not in training_eligible.columns:
            training_eligible[col] = "(synthetic -- not available)"
else:
    assert_seasons_allowed(sorted(df["season"].dropna().unique().tolist()))
    print("Confirmed: no 2025 data present in the loaded dataset.")
    training_eligible = df[df["eligible_for_training"].astype(bool)].copy()

train_df = training_eligible[training_eligible["season"].isin(TRAIN_SEASONS)]
val_df = training_eligible[training_eligible["season"].isin(VALIDATION_SEASONS)]

print(f"Training rows ({TRAIN_SEASONS}): {len(train_df)}")
print(f"Validation rows ({VALIDATION_SEASONS}): {len(val_df)}")
assert len(train_df) > 0, "No training rows available -- cannot proceed."
assert len(val_df) > 0, "No validation rows available -- cannot proceed."
assert train_df["season"].isin(TRAIN_SEASONS).all()
assert val_df["season"].isin(VALIDATION_SEASONS).all()
assert set(train_df.index).isdisjoint(val_df.index), (
    "Training and validation rows must never overlap."
)

Confirmed: no 2025 data present in the loaded dataset.


Training rows ((2021, 2022, 2023)): 364311
Validation rows ((2024,)): 122132


## Train the unweighted baseline on 2021-2023 ONLY

**The probability below comes from the `unweighted` model (`class_weight=None`) -- the variant selected as the Contact Luck probability baseline in `03_calibration.ipynb` after confirming it is far better calibrated than the `class_weight="balanced"` comparison model.** The training call is explicit about `class_weight=None` here even though it's the default, so that choice is visible directly in this notebook.

In [4]:
trained = train_model(train_df, class_weight=None)
print("Variant:", trained.variant)
print("class_weight:", trained.class_weight)
print("Numeric features:", trained.numeric_features)
print("Categorical features:", trained.categorical_features)

Variant: unweighted_probability_baseline
class_weight: None
Numeric features: ['launch_speed', 'launch_angle', 'spray_angle_approx', 'hit_distance_sc']
Categorical features: ['bb_type', 'stand']


## Select the example play from 2024 ONLY

Chosen as the first `home_run` row in the 2024 validation split (if any) purely for an illustrative example with a clearly favorable outcome -- not because it was cherry-picked for a large raw-luck value across many candidates. Falls back to the first validation row if no home run is present. **This row was never seen during training.**

In [5]:
home_run_candidates = val_df[val_df["outcome_class"] == "home_run"]
example = (home_run_candidates if not home_run_candidates.empty else val_df).iloc[[0]].copy()
observed_outcome = str(example["outcome_class"].iloc[0])
example_season = int(example["season"].iloc[0])

assert example_season in VALIDATION_SEASONS, "Example must come from a validation season."
assert example.index[0] not in train_df.index, "Example must never be a training row."

feature_cols = trained.numeric_features + trained.categorical_features
proba_df = predict_proba_ordered(trained, example[feature_cols])
probabilities = {cls: float(proba_df.iloc[0][cls]) for cls in CLASS_ORDER}

## Play details

In [6]:
def _display_value(row, col):
    if col in row.index and pd.notna(row[col]):
        return row[col]
    return "(not available)"

row = example.iloc[0]
print(f"Season:                  {example_season}")
print(f"Game date:               {_display_value(row, 'game_date')}")
print(f"Batter ID:               {_display_value(row, 'batter')}")
print(f"Pitcher ID:              {_display_value(row, 'pitcher')}")
# NOTE: Statcast's `player_name` field identifies the PITCHER, not the batter
# (verified against real data: a fixed pitcher ID maps to one `player_name` across
# many different batter IDs). There is no batter-name field in this schema.
print(f"Pitcher name:            {_display_value(row, 'player_name')}")
print(f"Event description:       {_display_value(row, 'description')}")
print(f"Actual outcome:          {observed_outcome}")

Season:                  2024
Game date:               2024-09-26
Batter ID:               592669
Pitcher ID:              571578
Pitcher name:            Corbin, Patrick
Event description:       hit_into_play
Actual outcome:          home_run


## Scoring walkthrough

In [7]:
expected_value = compute_expected_value(probabilities, value_map=DEFAULT_VALUE_MAP)
actual_value = DEFAULT_VALUE_MAP[observed_outcome]
raw_luck = compute_raw_luck(probabilities, observed_outcome, value_map=DEFAULT_VALUE_MAP)
public_score = raw_luck_to_public_score(raw_luck)
confidence = compute_confidence(row)

print("Predicted probability distribution (unweighted baseline):")
for cls in CLASS_ORDER:
    print(f"  {cls:>9s}: {probabilities[cls]:.3f}")
print(f"P(observed result):             {probabilities[observed_outcome]:.3f}")
print(f"Expected ordinal value:         {expected_value:.3f}")
print(f"Actual ordinal value:           {actual_value:.3f}")
print(f"Preliminary raw luck (additive):{raw_luck:+.3f}")
print(f"Preliminary public score:       {public_score:+.1f} / 100 (NOT additive)")
print(f"Data-completeness label:        {confidence.label}")
print(f"Data-completeness detail:       {confidence}")

Predicted probability distribution (unweighted baseline):
        out: 0.038
     single: 0.001
     double: 0.014
     triple: 0.003
   home_run: 0.944
P(observed result):             0.944
Expected ordinal value:         3.814
Actual ordinal value:           4.000
Preliminary raw luck (additive):+0.186
Preliminary public score:       +9.3 / 100 (NOT additive)
Data-completeness label:        medium_data_completeness
Data-completeness detail:       ConfidenceReport(required_fields_present_pct=71.42857142857143, missing_optional_fields=('venue', 'sprint_speed'), spray_direction_available=False, park_available=False, alignment_fields_available=True, fully_eligible=True, label='medium_data_completeness')


## Explicit provisional disclaimer

- The ordinal value map (`out=0, single=1, double=2, triple=3, home_run=4`) is a **Version 0.1 research placeholder**, not a validated run-value model.
- The public score's `tanh`-based mapping to [-100, 100] is a **provisional convenience choice**, not an empirically calibrated scientific mapping, and is **NOT additive** -- do not sum public scores across plays. It has not been fit against any historical distribution of raw-luck values.
- The confidence label reflects **observable data completeness only** -- it is not a statistical uncertainty estimate, and it does not affect (and should never affect) the raw-luck or public-score values above.
- Contact luck as computed here does **not yet separate out weather, park effects, exact defensive positioning, defensive execution, or baserunning decisions/execution** -- see README.md "Future work".
- The `unweighted` model itself has known, small remaining calibration imperfections (see notebook 03) -- this is a Version 0.1 research artifact, not a validated statistic.

> **This repository is a research prototype, not a validated public baseball statistic.** See `README.md` and `CLAUDE.md` for full scope and limitations.